# 模型选择


## 数据
### 数据复杂度
- 样本个数
- 特征维度
- 时间、空间结构
- 多样性
- 噪声水平

### 数据划分
- **训练集**: 用于训练模型
- **验证集**: 用于模型选择、调参和早停
- **测试集**: 用于评估模型的泛化能力

### K-折交叉验证
将训练集划分为 K 个子集
- for i = 1, 2, ..., K:
    - 将第 i 个子集作为验证集，其余 K-1 个子集作为训练集
    - 训练模型并评估性能

常用 k = 5 或 10

## 拟合效果
### 数据与模型
|          | 简单数据 | 复杂数据 |
|----------|----------|----------|
| 简单模型 | 正常     | 欠拟合   |
| 复杂模型 | 过拟合   | 正常     |

### VC 维数
对于一个分类模型，其 VC 维数等于一个最大数据集的大小，使得该数据集可以被模型完全拟合。

### 评估指标
- **训练误差**: 模型在训练集上的误差
- **泛化误差**: 模型在训练集以外数据上的误差

良好的模型应当满足
- 泛化误差低
- 训练误差与泛化误差相差不大


# 权重衰退


## 概念
在损失函数中加入L2范数惩罚项，在训练过程中让模型偏好较小的权重值。抑制模型过拟合、提高泛化能力

## L2范数惩罚项
也称均方范数
$$
\Omega(\theta) = \frac{1}{2} \sum_{i=1}^{n} \theta_i^2
$$
记为 $$\Omega(\theta) = \frac{1}{2} ||\theta||_2^2$$

## 损失函数
$$
l(w) = l_0(w) + \frac{\eta}{2} ||w||_2^2
$$
其中 $\eta$ 为正则化系数，控制惩罚项的权重

## 计算梯度
$$
\begin{aligned}
\frac{\delta l(w)}{\delta w}
&= \frac{\delta l_0(w)}{\delta w} + \frac{\eta}{2} \frac{\delta ||w||_2^2}{\delta w} \\
&= \frac{\delta l_0(w)}{\delta w} + \eta w
\end{aligned}
$$
梯度下降更新后的参数
$$
\begin{aligned}
w_{t+1}
&= w_t - \alpha(\frac{\delta l(w)}{\delta w}) \\
&= w_t - \alpha(\frac{\delta l_0(w)}{\delta w} + \eta w) \\
&= (1 - \alpha \eta) w_t - \alpha \frac{\delta l_0(w)}{\delta w}
\end{aligned}
$$
其中通常 $\alpha \eta < 1$，因此 $(1 - \alpha \eta) < 1$，即每次迭代后权重都会衰减。

## 几何理解
在两个曲面（损失函数曲面、惩罚项曲面）的合成曲面上梯度下降，最终找到损失和惩罚项的平衡点

![两个曲面等高线](https://i-blog.csdnimg.cn/blog_migrate/ca2bde3f7fb73ece67e226e615a9fb81.png)


# 丢弃法


## 概念
在训练过程中随机丢弃一部分神经元，使得模型不依赖于某些特定的神经元，从而提高模型的泛化能力。

## 丢弃
对每个元素进行如下扰动
$$
x_i' =
\begin{cases}
0, & \text{以概率 } p \\
\cfrac{x_i}{1-p}, & \text{以概率 } 1-p
\end{cases}
$$
> [!NOTE]
> 让一部分权重退出，另一部分权重放大，训练“子神经网络”，避免模型依赖某一两个突出神经元。

不改变期望
$$
E[x_i'] = p \cdot 0 + (1-p) \cdot \cfrac{x_i}{1-p} = x_i
$$


# 数值稳定性


## 梯度问题
损失 $l$ 对权重 $W^t$ 的梯度为
$$ \cfrac{\delta l}{\delta W^t} = \cfrac{\delta l}{\delta h^d} \cdot \cfrac{\delta h^d}{\delta h^{d-1}} \cdots \cfrac{\delta h^{t+1}}{\delta h^t} \cdot \cfrac{\delta h^{t}}{\delta W^t} $$

$h_i$ 为向量，故 $\cfrac{\delta h^{i+1}}{\delta h^i}$ 为矩阵，则计算梯度时做了 $d-t$ 次矩阵乘法

- **梯度爆炸**: $1.5^{100} \approx 4.0 \times 10^{17}$，梯度过大，权重更新过大，模型不收敛
- **梯度消失**: $0.5^{100} \approx 7.9 \times 10^{-31}$，梯度过小，权重更新过小，模型无法学习

### MLP 为例
省略偏置，每一层与上一层的关系为
$$ h^t = \sigma(W^t h^{t-1}) $$

计算对上一层的梯度
$$ \cfrac{\delta h^t}{\delta h^{t-1}} = \cfrac{\delta \sigma(W^t h^{t-1})}{\delta h^{t-1}} = diag(\sigma'(W^t h^{t-1})) \cdot (W^t)^T $$

- **$diag$**: $\sigma$ 激活函数按元素，求导是对角阵（跨元素偏导为 0）
- **$(W^t)^T$**: $W^t h^{t-1}$ 对 $h^{t-1}$ 逐项偏导的结果

梯度逐层累乘
$$ grad^t = \prod_{i=t+1}^{d} \cfrac{\delta h^i}{\delta h^{i-1}} = \prod_{i=t+1}^{d} diag(\sigma'(W^i h^{i-1})) \cdot (W^i)^T $$

#### ReLU 激活
$$ \sigma(x) = \max(x, 0) $$
$$ \sigma'(x) =
\begin{cases}
1, & x > 0 \\
0, & x \leq 0
\end{cases}
$$

$diag(\sigma'(W^i h^{i-1}))$ 是只含 0、1 的对角阵

- $\Rightarrow$ 左乘 $diag(\sigma'(W^i h^{i-1}))$ 的含义是“屏蔽某些列，保留另一些列”

- $\Rightarrow$ $grad^t$ 值主要来自 $\prod_{i=t+1}^{d} (W^i)^T$

若 $d-t$ 很大，且 $W^i$ 的元素 $\gt 1$，则 $grad^t$ 很大，梯度爆炸

#### Sigmoid 激活
$$ \sigma(x) = \cfrac{1}{1 + e^{-x}} $$
$$ \sigma'(x) = \sigma(x)(1 - \sigma(x)) $$

$x$ 较大或较小时，都会导致 $\sigma'(x)$ 接近 0

- $\Rightarrow$ $diag(\sigma'(W^i h^{i-1}))$ 对角线上多数元素很小

- $\Rightarrow$ $grad^t$ 是接近 $d-t$ 个小元素的乘积

若 $d-t$ 很大，则 $grad^t$ 很小，梯度消失




## 权重初始化
让初始权重的数值在一个合理范围中
- 将每层的输入、输出和梯度看作随机变量
- 让它们的均值、方差保持一致

### MLP 为例
做如下假设
- $w^t_{i,j}$ 是独立同分布，则 $\mathbb{E}[w^t_{i,j}] = 0$，$\mathbb{D}[w^t_{i,j}] = \gamma_t\text{（常数）}$
- $h^{t-1}_i$ 独立于 $w^t_{i,j}$
- 暂不考虑激活函数，$h^t = W^t h^{t-1}$，则
$$
\mathbb{E}[h^t_i] = \mathbb{E} \left[\sum_{j=1}^{n_{t-1}} w^t_{i,j} h^{t-1}_j \right] = \sum_{j=1}^{n_{t-1}} \mathbb{E}[w^t_{i,j}] \mathbb{E}[h^{t-1}_j] = 0
$$

正向方差
$$
\begin{aligned}
\mathbb{D}[h^t_i]
&= \mathbb{E}[(h^t_i)^2] - (\mathbb{E}[h^t_i])^2 \\
&= \mathbb{E}\left[(\sum_{j=1}^{n_{t-1}} w^t_{i,j} h^{t-1}_j)^2\right] \\
&= \mathbb{E}\left[\sum_{j=1}^{n_{t-1}} (w^t_{i,j})^2 (h^{t-1}_j)^2 + \sum_{j_1 \neq j_2} w^t_{i,j_1} h^{t-1}_{j_1} w^t_{i,j_2} h^{t-1}_{j_2}\right] \\
&= \sum_{j=1}^{n_{t-1}} \mathbb{E}[(w^t_{i,j})^2] \mathbb{E}[(h^{t-1}_j)^2] \\
&= \sum_{j=1}^{n_{t-1}} \mathbb{D}[w^t_{i,j}] \mathbb{D}[h^{t-1}_j]
&= n_{t-1} \gamma_t \mathbb{D}[h^{t-1}_j]
\end{aligned}
$$
- 希望输入输出方差相等，则 $n_{t-1} \gamma_t = 1$

- 反向方差同理推导出 $n_{t} \gamma_t = 1$

二者不能同时满足，Xavier 初始化取平均值，使 $\gamma_t = \cfrac{2}{n_{t-1} + n_t}$
- 正态分布 $\mathcal{N}(0, \sqrt{\gamma_t}) = \mathcal{N}\left(0, \sqrt{\cfrac{2}{n_{t-1} + n_t}}\right)$
- 均匀分布 $\mathcal{U}(-\sqrt{3\gamma_t}, \sqrt{3\gamma_t}) = \mathcal{U}\left(-\sqrt{\cfrac{6}{n_{t-1} + n_t}}, \sqrt{\cfrac{6}{n_{t-1} + n_t}}\right)$


## 激活函数
### 稳定性条件
假设线性的激活函数 $\sigma(x) = \alpha x + \beta$，

并且有 $h' = W^t h^{t-1}, h^t = \sigma(h')$，则
$$
\mathbb{E}[h^t_i] = \alpha \mathbb{E}[h'_i] + \beta = \beta
$$
$$
\begin{aligned}
\mathbb{D}[h^t_i]
&= \mathbb{E}[(h^t_i)^2] - (\mathbb{E}[h^t_i])^2 \\
&= \mathbb{E}[(\alpha h'_i + \beta)^2] - \beta^2 \\
&= \alpha^2 \mathbb{E}[(h'_i)^2] + 2\alpha\beta \mathbb{E}[h'_i] + \beta^2 - \beta^2
&= \alpha^2 \mathbb{D}[h'_i]
\end{aligned}
$$
希望激活前后均值为0、方差相等，需要 $\beta = 0, \alpha = 1$，即 $\sigma(x) = x$。

### 非线性激活函数
常用的激活函数泰勒展开
$$
\begin{aligned}
&relu(x)  = 0 + x, for \space x \geq 0 \\
&tanh(x) = 0 + x - \frac{x^3}{3} + \mathcal{O}(x^5) \\
&sigmoid(x) = \frac{1}{2} + \frac{x}{4} - \frac{x^3}{48} + \mathcal{O}(x^5)
\end{aligned}
$$
- $relu$ 和 $tanh$ 在原点附近近似 $\sigma(x) = x$，满足稳定性条件

- $sigmoid$ 在原点附近近似 $\sigma(x) = \frac{1}{2} + \frac{x}{4}$，需要调整为 $4\times sigmoid(x) - 2$ 才能满足稳定性条件

